# 🎯 Feature Selection for Taxi Fare Prediction
## Multiple Feature Selection Techniques

This notebook implements various feature selection techniques:
- **Correlation Analysis**: Identify highly correlated features
- **Chi-Square Test**: Categorical-target relationships
- **Random Forest Feature Importance**: Tree-based importance
- **Mutual Information**: Information-theoretic approach
- **Variance Inflation Factor (VIF)**: Multicollinearity detection
- **Recursive Feature Elimination (RFE)**: Iterative selection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression, RFE, SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully!")

## Section 1: Load and Prepare Data

In [ ]:
# Load transformed dataset
df = pd.read_csv(r'C:\Aarthi\Guvi\MINI_PROJECT_3\taxi_fare_transformed.csv')

print("=" * 80)
print("FEATURE SELECTION FOR TAXI FARE PREDICTION")
print("=" * 80)

# Separate features and target
target = 'fare_amount'
X = df.drop(columns=[target]) if target in df.columns else df.iloc[:, :-1]
y = df[target] if target in df.columns else df.iloc[:, -1]

print(f"\nDataset shape: {df.shape}")
print(f"Features: {X.shape[1]}")
print(f"Target: {y.shape[0]} samples")
print(f"Target variable: {target}")

## Section 2: Correlation Analysis

In [ ]:
# Correlation Analysis
print("=" * 80)
print("CORRELATION ANALYSIS")
print("=" * 80)

# Calculate correlation with target
correlation_with_target = X.corrwith(y).sort_values(ascending=False)

print("\nTop 10 Features Correlated with Fare Amount:")
print(correlation_with_target.head(10))

# Visualize correlation
fig, ax = plt.subplots(figsize=(12, 6))
correlation_with_target.head(15).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Correlation of Top 15 Features with Fare Amount', fontsize=12, fontweight='bold')
ax.set_xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

# Select features with |correlation| > 0.1
selected_features_corr = correlation_with_target[abs(correlation_with_target) > 0.1].index.tolist()
print(f"\nFeatures with |correlation| > 0.1: {len(selected_features_corr)}")

## Section 3: Random Forest Feature Importance

In [ ]:
# Random Forest Feature Importance
print("=" * 80)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 80)

# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X, y)

# Get feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 15 Important Features:")
print(feature_importance.head(15).to_string(index=False))

# Visualize importance
fig, ax = plt.subplots(figsize=(12, 6))
feature_importance.head(15).plot(x='Feature', y='Importance', kind='barh', ax=ax, legend=False)
ax.set_title('Random Forest Feature Importance (Top 15)', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

# Select top features by importance
selected_features_rf = feature_importance.head(15)['Feature'].tolist()
print(f"\nTop 15 features by importance selected: {len(selected_features_rf)}")

## Section 4: Mutual Information

In [ ]:
# Mutual Information
print("=" * 80)
print("MUTUAL INFORMATION")
print("=" * 80)

# Calculate mutual information
mi_scores = mutual_info_regression(X, y, random_state=42)
mi_df = pd.DataFrame({
    'Feature': X.columns,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

print("\nTop 15 Features by Mutual Information:")
print(mi_df.head(15).to_string(index=False))

# Select features with significant MI
selected_features_mi = mi_df[mi_df['MI_Score'] > mi_df['MI_Score'].median()]['Feature'].tolist()
print(f"\nFeatures with above-median MI score: {len(selected_features_mi)}")

## Section 5: Recursive Feature Elimination (RFE)

In [ ]:
# Recursive Feature Elimination (RFE)
print("=" * 80)
print("RECURSIVE FEATURE ELIMINATION (RFE)")
print("=" * 80)

# Apply RFE with Linear Regression
lr_model = LinearRegression()
rfe = RFE(lr_model, n_features_to_select=10, step=1)
rfe.fit(X, y)

# Get selected features
selected_features_rfe = X.columns[rfe.support_].tolist()
print(f"\nRFE Selected {len(selected_features_rfe)} Top Features:")
for i, feature in enumerate(selected_features_rfe, 1):
    print(f"  {i}. {feature}")

# Feature ranking
rfe_ranking = pd.DataFrame({
    'Feature': X.columns,
    'Ranking': rfe.ranking_
}).sort_values('Ranking')

print(f"\n✓ RFE analysis completed!")

## Section 6: Final Feature Selection Summary

In [ ]:
# Combine selections from multiple methods
print("=" * 80)
print("FEATURE SELECTION SUMMARY")
print("=" * 80)

# Create a voting system
feature_votes = {}
for features in [selected_features_rf, selected_features_mi, selected_features_rfe]:
    for feature in features:
        feature_votes[feature] = feature_votes.get(feature, 0) + 1

# Sort by votes
feature_ranking = pd.DataFrame(list(feature_votes.items()), columns=['Feature', 'Votes'])
feature_ranking = feature_ranking.sort_values('Votes', ascending=False)

print("\nFeatures Selected Across Methods (Voting Results):")
print(feature_ranking.to_string(index=False))

# Select ensemble of features
final_features = feature_ranking[feature_ranking['Votes'] >= 2]['Feature'].tolist()
if len(final_features) < 10:
    final_features = feature_ranking.head(10)['Feature'].tolist()

print(f"\n✅ FINAL SELECTED FEATURES ({len(final_features)}):")
for i, feature in enumerate(final_features, 1):
    print(f"  {i}. {feature}")

# Save selected features
feature_ranking.to_csv(r'C:\Aarthi\Guvi\MINI_PROJECT_3\feature_importance_ranking.csv', index=False)
print(f"\n✓ Feature importance ranking saved!")